# EXO_05_VALIDATION_10F — Test Pipeline Alchemist (10 frames)

```
╔══════════════════════════════════════════════════════════════════════════════╗
║         ALCHEMIST LAB — VALIDATION 10 FRAMES — AVANT PRODUCTION             ║
║                                                                              ║
║   Setup → Sync → Phantom → Marshal → SENTINEL → Run 10F → Check → SENTINEL  ║
║                                                                              ║
║   But : Valider le pipeline sur 10 frames avant batch complet                ║
║   Output : OUT_VALIDATION_10F/ (isole de OUT_FINAL_FRAMES)                  ║
╚══════════════════════════════════════════════════════════════════════════════╝
```

**Lancer ce notebook AVANT EXO_05_PRODUCTION pour valider le pipeline.**

## 1. Setup

In [ ]:
import os
import sys
import json
import shutil
from pathlib import Path

from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Drive deja monte')

DRIVE_ROOT  = Path('/content/drive/MyDrive/EXODUS_V2')
# DRIVE_ROOT = Path('/home/EXODUS-V2')  # Local dev

UNIT_ROOT       = DRIVE_ROOT / '05_ALCHEMIST_LAB'
CODEBASE        = UNIT_ROOT / 'CODEBASE'
IN_RAW_FRAMES   = UNIT_ROOT / 'IN_RAW_FRAMES'
IN_SOURCE_REF   = UNIT_ROOT / 'IN_SOURCE_REF'
OUT_VALIDATION  = UNIT_ROOT / 'OUT_VALIDATION_10F'

PRESET          = 'cinema_fusion'  # cinema_fusion | subtle_blend | neon_blast | raw_match
N_FRAMES        = 10               # Nombre de frames a tester
SCENE_ID        = 1                # Scene a utiliser (1, 2 ou 3)

sys.path.insert(0, str(CODEBASE))

!pip install -q numpy opencv-python-headless Pillow tqdm

print(f'Drive Root     : {DRIVE_ROOT}')
print(f'IN_RAW_FRAMES  : {IN_RAW_FRAMES}')
print(f'IN_SOURCE_REF  : {IN_SOURCE_REF}')
print(f'OUT_VALIDATION : {OUT_VALIDATION}')
print(f'Preset         : {PRESET}')
print(f'N Frames       : {N_FRAMES}')
print(f'Scene ID       : {SCENE_ID}')

## ⚡ GPU Check

In [ ]:
# ── GPU CHECK — U05 ──────────────────────────────────────────────────
# [VULKAN_FORGE] Detection GPU + activation CUDA OpenCV si disponible
import subprocess, os

# 1. nvidia-smi
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                             '--format=csv,noheader'],
                            capture_output=True, text=True, timeout=10)
    gpu_info = result.stdout.strip()
    GPU_AVAILABLE = result.returncode == 0 and bool(gpu_info)
except Exception:
    GPU_AVAILABLE = False
    gpu_info = 'nvidia-smi absent'

# 2. OpenCV CUDA
import cv2
OPENCV_CUDA = cv2.cuda.getCudaEnabledDeviceCount() > 0

# 3. Flag global USE_CUDA
USE_CUDA = GPU_AVAILABLE and OPENCV_CUDA

print('=== GPU Check ===')
if GPU_AVAILABLE:
    print(f'  GPU detecte    : {gpu_info}')
else:
    print('  GPU            : ABSENT — mode CPU')
print(f'  OpenCV CUDA    : {"OUI" if OPENCV_CUDA else "NON"}')
print(f'  USE_CUDA       : {USE_CUDA}')
print(f'  Mode actif     : {"GPU (CUDA accelere)" if USE_CUDA else "CPU"}')

if USE_CUDA:
    print('  [VULKAN] CUDA active — operations OpenCV accelerees GPU')
else:
    print('  [VULKAN] CPU mode — pipeline OK, plus lent sans GPU')
    print('  CONSEIL : Runtime > Changer le type de runtime > GPU (T4)')


## ⚙️ Sync Codebase depuis GitHub

In [ ]:
# === SYNC CODEBASE depuis GitHub (applique les derniers fixes) ===
# [VULKAN_FORGE] Garantit que Colab execute toujours la derniere version
import subprocess

REPO_TMP = Path('/tmp/exodus-u05-sync')
if REPO_TMP.exists():
    shutil.rmtree(REPO_TMP)

subprocess.run(
    ['git', 'clone', '--depth', '1',
     'https://github.com/kioka8877-ux/EXODUS-V2.git', str(REPO_TMP)],
    check=True, capture_output=True
)

SRC = REPO_TMP / '05_ALCHEMIST_LAB' / 'CODEBASE'
DST = CODEBASE
DST.mkdir(parents=True, exist_ok=True)

synced = 0
for f in SRC.iterdir():
    if f.is_file():
        shutil.copy2(str(f), str(DST / f.name))
        synced += 1

print(f'[VULKAN] CODEBASE sync OK — {synced} fichiers depuis GitHub → {DST}')

## ⚡ Phantom Link

In [ ]:
# Cree les phantom links (lit directement depuis OUT/ de la fregate source)
!python "{DRIVE_ROOT}/EXO_MARSHAL.py" --unit U05 --mode link --drive-root {DRIVE_ROOT} --verbose

## 2. Marshal In-Check + Pre-flight

In [ ]:
print('=== Marshal In-Check ===')
!python "{DRIVE_ROOT}/EXO_MARSHAL.py" --unit U05 --mode check-in --drive-root {DRIVE_ROOT} --verbose

print('\n=== Pre-flight ===')

# Scan frames disponibles
all_frames = []
if IN_RAW_FRAMES.exists():
    all_frames = sorted(IN_RAW_FRAMES.glob('*.png')) + sorted(IN_RAW_FRAMES.glob('*.exr'))
    print(f'  Total frames disponibles : {len(all_frames)}')
    for f in all_frames[:5]:
        size_kb = f.stat().st_size / 1024
        print(f'    {f.name} ({size_kb:.0f} KB)')
    if len(all_frames) > 5:
        print(f'    ... et {len(all_frames) - 5} autres')
else:
    print('  IN_RAW_FRAMES : ABSENT')

# Scan video source
source_video = None
if IN_SOURCE_REF.exists():
    videos = list(IN_SOURCE_REF.glob('*.mp4')) + list(IN_SOURCE_REF.glob('*.mov'))
    if videos:
        source_video = videos[0]
        size_mb = source_video.stat().st_size / (1024 * 1024)
        print(f'  Video source : {source_video.name} ({size_mb:.1f} MB)')
    else:
        print('  Video source : ABSENTE (match_color/grain/sharpness desactives)')
else:
    print('  IN_SOURCE_REF : ABSENT')

# Scan plan production
plan_candidates = (list(UNIT_ROOT.glob('**/PRODUCTION_PLAN*.json'))
                   + list(UNIT_ROOT.glob('**/PRODUCTION_PLAN*.JSON')))
PLAN_PATH = plan_candidates[0] if plan_candidates else None
print(f'  Plan : {PLAN_PATH.name if PLAN_PATH else "ABSENT"}')

ready = len(all_frames) >= N_FRAMES
print(f'\n  Frames suffisantes ({N_FRAMES} requis) : {"OUI" if ready else "NON"}')
print(f'  Pret pour validation : {"OUI" if ready else "NON — relancer apres correction"}')

## 🛡️ SENTINEL — Pre-Check U05

In [ ]:
# ── SENTINEL Pre-Check — U05 ──────────────────────────────────────────
# [VULKAN_FORGE] Systeme immunitaire — bloque si fregate corrompue
sys.path.insert(0, str(DRIVE_ROOT / 'SENTINEL_CORE' / 'CODEBASE'))

try:
    from sentinel_core import Sentinel
    sentinel = Sentinel(base_dir=str(DRIVE_ROOT / 'SENTINEL_CORE'))
    sentinel_precheck = sentinel.run(
        fregate='U05',
        frames_dir=str(IN_RAW_FRAMES)
    )
    print(f'[SENTINEL PRE-CHECK] Verdict : {sentinel_precheck["verdict"]}')
    if sentinel_precheck['verdict'] == 'FAIL':
        print('\n[SENTINEL] BLOCAGE — Lire le prompt Vulkan ci-dessous :')
        print(sentinel_precheck.get('prompt_vulkan', 'Aucun prompt genere'))
        raise RuntimeError('[SENTINEL] Fregate U05 bloquee — corriger avant de lancer.')
    else:
        print('[SENTINEL] Pre-check OK — U05 autorisee a demarrer.')
except ImportError:
    print('[SENTINEL] WARN — sentinel_core non disponible, pre-check ignore.')

## 3. Preparation — Selection 10 frames

In [ ]:
import cv2
import numpy as np

# Creer dossier temporaire avec les N_FRAMES selectionnees
TEST_FRAMES_DIR = Path('/tmp/exodus_u05_test_frames')
if TEST_FRAMES_DIR.exists():
    shutil.rmtree(TEST_FRAMES_DIR)
TEST_FRAMES_DIR.mkdir(parents=True)

# Selectionner N_FRAMES depuis IN_RAW_FRAMES
# Strategie : prendre les N premiers frames de la scene cible
selected = []
for f in sorted(all_frames):
    name = f.stem.lower()
    # Detecter scene
    sid = 1
    if '_scene_' in name:
        try:
            sid = int(name.split('_scene_')[1].split('_')[0])
        except (ValueError, IndexError):
            pass
    elif 'scene' in name:
        try:
            idx = name.index('scene')
            num_str = ''
            for c in name[idx + 5:]:
                if c.isdigit():
                    num_str += c
                elif num_str:
                    break
            if num_str:
                sid = int(num_str)
        except (ValueError, IndexError):
            pass
    if sid == SCENE_ID and len(selected) < N_FRAMES:
        selected.append(f)

# Fallback : si pas assez depuis la scene, prendre les premiers
if len(selected) < N_FRAMES:
    print(f'  WARN : scene {SCENE_ID} a {len(selected)} frames — fallback premiers {N_FRAMES}')
    selected = sorted(all_frames)[:N_FRAMES]

# Copier dans dossier temp
for i, f in enumerate(selected):
    dst = TEST_FRAMES_DIR / f'{f.stem}_test{f.suffix}'
    shutil.copy2(str(f), str(dst))

print(f'  {len(selected)} frames selectionnees depuis scene {SCENE_ID}')
for f in selected:
    size_kb = f.stat().st_size / 1024
    # Lire dimensions
    img = cv2.imread(str(f), cv2.IMREAD_UNCHANGED)
    if img is not None:
        h, w = img.shape[:2]
        print(f'    {f.name:30s} | {w}x{h} | {img.dtype} | {size_kb:.0f} KB')
    else:
        print(f'    {f.name:30s} | ERREUR lecture')

print(f'\n  Dossier test : {TEST_FRAMES_DIR}')

## 4. Execution — Pipeline 10 frames

In [ ]:
OUT_VALIDATION.mkdir(parents=True, exist_ok=True)

cmd  = f'python "{CODEBASE}/EXO_05_ALCHEMIST.py"'
cmd += f' --drive-root "{DRIVE_ROOT}"'
cmd += f' --preset {PRESET}'
cmd += f' --render-dir "{TEST_FRAMES_DIR}"'
cmd += f' --output-dir "{OUT_VALIDATION}"'

if source_video:
    cmd += f' --source-video "{source_video}"'
else:
    cmd += ' --skip-match --skip-grain --skip-sharpness'

if PLAN_PATH:
    cmd += f' --production-plan "{PLAN_PATH}"'

cmd += ' -v'

print('=== Commande ===')
print(cmd)
print('\n=== Execution ===')
!{cmd}

## 5. Verification Output

In [ ]:
import json as _json

print('=== Output Check ===')

outputs = sorted(OUT_VALIDATION.glob('*.png')) if OUT_VALIDATION.exists() else []
print(f'  Frames produites : {len(outputs)} / {N_FRAMES} attendues')

ok_count = 0
warn_count = 0
for f in outputs:
    size_kb = f.stat().st_size / 1024
    img = cv2.imread(str(f), cv2.IMREAD_UNCHANGED)
    if img is not None:
        h, w = img.shape[:2]
        is_16bit = img.dtype == np.uint16
        tag = 'OK 16-bit' if is_16bit else f'WARN {img.dtype}'
        status = 'OK' if is_16bit else 'WARN'
        if status == 'OK':
            ok_count += 1
        else:
            warn_count += 1
        print(f'  [{status}] {f.name:35s} | {w}x{h} | {tag} | {size_kb:.0f} KB')
    else:
        warn_count += 1
        print(f'  [ERR] {f.name:35s} | ERREUR lecture')

# Lire rapport JSON si present
report_path = OUT_VALIDATION / 'alchemist_report.json'
if report_path.exists():
    with open(report_path) as f:
        report = _json.load(f)
    s = report.get('summary', {})
    print(f'\n=== Rapport Alchemist ===')
    print(f'  Status          : {s.get("status", "?")}')  
    print(f'  Frames traitees : {s.get("total_frames_processed", 0)}')
    print(f'  Erreurs         : {s.get("total_frames_failed", 0)}')
    print(f'  Temps total     : {s.get("total_time_seconds", 0):.1f}s')
    frames_ok = s.get('total_frames_processed', 0)
    if frames_ok > 0:
        avg = s.get('total_time_seconds', 0) / frames_ok
        print(f'  Moyenne         : {avg:.2f}s/frame')
    print(f'  Pipeline        : {" → ".join(report.get("pipeline", []))}')

# Verdict global
print(f'\n=== Verdict ===')
if len(outputs) == N_FRAMES and warn_count == 0:
    print(f'  PASS — {N_FRAMES}/{N_FRAMES} frames OK 16-bit')
    print('  U05 pret pour PRODUCTION')
elif len(outputs) > 0:
    print(f'  WARN — {ok_count} OK, {warn_count} WARN/ERR sur {N_FRAMES} frames')
    print('  Investiguer avant PRODUCTION')
else:
    print('  FAIL — 0 frames produites')
    print('  Verifier les logs ci-dessus')

## 🛡️ SENTINEL — Post-Run U05

In [ ]:
# ── SENTINEL Post-Run — U05 ──────────────────────────────────────────
# [VULKAN_FORGE] Validation finale + injection ledger
try:
    from sentinel_core import Sentinel
    sentinel_rapport = sentinel.run(
        fregate='U05',
        frames_dir=str(OUT_VALIDATION)
    )
    verdict = sentinel_rapport['verdict']
    print(f'[SENTINEL POST-RUN] Verdict final : {verdict}')
    print(f'  Injections ledger   : {sentinel_rapport["ledger_injections"]}')
    print(f'  Fichiers sauvegardes: {sentinel_rapport["fichiers_sauvegardes"]}')
    if verdict == 'FAIL':
        print('\n[SENTINEL] ACTION REQUISE — Copier le prompt Vulkan dans Claude :')
        print(sentinel_rapport.get('prompt_vulkan', 'Aucun prompt'))
    else:
        print(f'[SENTINEL] U05 validation terminee — Ledger mis a jour.')
except (ImportError, NameError):
    print('[SENTINEL] WARN — sentinel_core non disponible, post-run ignore.')

print()
print('=' * 60)
print('   ALCHEMIST LAB — VALIDATION 10F TERMINEE')
print('   Si PASS → lancer EXO_05_PRODUCTION.ipynb')
print('=' * 60)